[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SekyungHan/ai-power-textbook-labs/blob/master/solutions/ch02_autoencoder_anomaly.ipynb)

# Ch 2: 오토인코더 기반 전력 수요 이상 패턴 검출 (정답)

# Ch02: 오토인코더 기반 전력 수요 이상 패턴 검출 (Solution)

이 실습에서는 **오토인코더(Autoencoder)**를 학습하여 정상적인 전력 수요 패턴을 학습한 뒤,
재구성 오차를 기반으로 **이상 패턴(명절, 설비 고장 등)**을 자동 탐지합니다.

In [ ]:
# Ch02: 오토인코더 기반 전력 수요 이상 패턴 검출
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'koreanize-matplotlib'])
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)

try:
    sys.path.insert(0, '..')
    from utils.style import set_style, COLORS
    set_style()
except:
    # 한글 폰트 fallback (utils.style 없는 환경)
    import platform as _pf
    _sys = _pf.system()
    if _sys == 'Windows':
        plt.rcParams['font.family'] = 'Malgun Gothic'
    elif _sys == 'Darwin':
        plt.rcParams['font.family'] = 'AppleGothic'
    else:
        try:
            import koreanize_matplotlib
        except ImportError:
            plt.rcParams['font.family'] = 'NanumGothic'
    plt.rcParams['axes.unicode_minus'] = False


## 1. 데이터: 시간별 전력 수요 시계열

90일간의 시간별 전력 수요 데이터를 합성합니다. 일간/주간 패턴에 노이즈를 더하고,
명절(수요 급감)과 설비 고장(스파이크) 이상을 주입합니다.

In [ ]:
n_days = 90
hours = np.arange(n_days * 24)

daily = 50 + 30 * np.sin(2 * np.pi * (hours - 6) / 24)
weekly = np.where((hours // 24) % 7 >= 5, -10, 0)
noise = np.random.normal(0, 3, len(hours))
demand = daily + weekly + noise

# 이상 주입
demand[48*24:50*24] *= 0.7       # 명절 (수요 급감)
demand[70*24+12:70*24+18] += 40  # 설비 고장 스파이크

# 시각화
plt.figure(figsize=(14, 4))
plt.plot(demand, linewidth=0.5)
plt.axvspan(48*24, 50*24, alpha=0.3, color='red', label='명절')
plt.axvspan(70*24+12, 70*24+18, alpha=0.3, color='orange', label='설비 고장')
plt.xlabel('시간')
plt.ylabel('전력 수요 (MW)')
plt.title('90일 전력 수요 시계열')
plt.legend()
plt.show()

## 2. 데이터 전처리 및 오토인코더 학습

24시간 단위 윈도우로 시퀀스를 분할하고, 처음 60일을 학습 데이터로 사용합니다.
오토인코더는 정상 패턴의 압축 표현을 학습합니다.

In [ ]:
window = 24
sequences = []
for i in range(0, len(demand) - window, window):
    sequences.append(demand[i:i+window])
sequences = np.array(sequences)

train_data = sequences[:60]
test_data = sequences[60:]

mu, sigma = train_data.mean(), train_data.std()
train_norm = (train_data - mu) / sigma
test_norm = (test_data - mu) / sigma

In [ ]:
class DemandAE(nn.Module):
    def __init__(self, input_dim=24, latent_dim=6):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

In [ ]:
model = DemandAE()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

X_train = torch.FloatTensor(train_norm)
for epoch in range(300):
    optimizer.zero_grad()
    recon = model(X_train)
    loss = criterion(recon, X_train)
    loss.backward()
    optimizer.step()

print(f"학습 완료 — 최종 재구성 오차: {loss.item():.6f}")

## 3. 이상 점수 산출 및 시각화

테스트 데이터의 재구성 오차를 이상 점수로 사용하고, 학습 데이터의 95 퍼센타일을 임계값으로 설정합니다.

In [ ]:
model.eval()
with torch.no_grad():
    X_test = torch.FloatTensor(test_norm)
    recon_test = model(X_test)
    anomaly_scores = ((X_test - recon_test) ** 2).mean(dim=1).numpy()

with torch.no_grad():
    recon_train = model(X_train)
    train_scores = ((X_train - recon_train) ** 2).mean(dim=1).numpy()
threshold = np.percentile(train_scores, 95)

anomalies = anomaly_scores > threshold
print(f"임계값: {threshold:.4f}")
print(f"탐지된 이상 일수: {anomalies.sum()} / {len(anomalies)}")

# 시각화
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

days = np.arange(60, 60 + len(anomaly_scores))
axes[0].bar(days, anomaly_scores, color=['red' if a else 'steelblue' for a in anomalies])
axes[0].axhline(y=threshold, color='orange', linestyle='--', label=f'임계값 = {threshold:.4f}')
axes[0].set_ylabel('이상 점수 (MSE)')
axes[0].set_title('일별 이상 점수')
axes[0].legend()

# 원본 vs 재구성 (이상일)
anomaly_idx = np.where(anomalies)[0]
if len(anomaly_idx) > 0:
    idx = anomaly_idx[0]
    axes[1].plot(test_norm[idx], label='원본', marker='o', markersize=3)
    axes[1].plot(recon_test[idx].numpy(), label='재구성', marker='s', markersize=3)
    axes[1].set_title(f'이상 탐지일 (Day {60+idx}): 원본 vs 재구성')
    axes[1].set_xlabel('시간 (h)')
    axes[1].legend()

plt.tight_layout()
plt.show()